# Environment Setup for Azure AI Foundry Workshop

This notebook will guide you through setting up your environment for the Azure AI Foundry workshop.

## Prerequisites
- Python 3.8 or later
- Azure subscription with AI services access
- Basic Python knowledge

## Azure Authentication Setup
First, we'll verify our Azure credentials and setup.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
import os

# Initialize Azure credentials
try:
    credential = DefaultAzureCredential()
    print("✓ Successfully initialized DefaultAzureCredential")
except Exception as e:
    print(f"× Error initializing credentials: {str(e)}")

## Initialize AI Project Client

> **Note:** Before proceeding, ensure you:
> 1. Copy your `.env.example` file to `.env`
> 2. Set `PROJECT_ENDPOINT` in your `.env` file
> 3. Have a Project already provisioned in Microsoft Foundry

You can find your **Project endpoint** on the [Microsoft Foundry](https://ai.azure.com) Home page (it looks like `https://<resource>.services.ai.azure.com/api/projects/<project>`).


## Understanding AIProjectClient

The AIProjectClient is a key component for interacting with Azure AI services that:

- **Manages Connections**: Lists and accesses Azure AI resources like OpenAI models
- **Handles Authentication**: Securely connects using Azure credentials  
- **Enables Model Access**: Provides interfaces to use AI models and deployments
- **Manages Project Settings**: Controls configurations for your Azure AI project

The client requires:
- A **project endpoint** (`PROJECT_ENDPOINT`, from the Foundry Home page)
- Valid Azure credentials (`DefaultAzureCredential`)


In [ ]:
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent
load_dotenv(parent_dir / '.env')

# Initialize the endpoint-based AIProjectClient (New Foundry)
try:
    client = AIProjectClient(
        endpoint=os.getenv("PROJECT_ENDPOINT"),
        credential=credential,
    )
    print("✓ Successfully initialized AIProjectClient")
except Exception as e:
    print(f"× Error initializing client: {str(e)}")

## Verify Access to Models and Connections
Let's verify we can access the resources specified in the [prerequisites](../README.md#-prerequisites).

The cell below uses the endpoint-based `AIProjectClient` to enumerate the connections defined in your Foundry project. Look for:
- An **Azure AI Search** connection for vector search and knowledge retrieval

> **Note:** In the New Foundry account model, the models you deploy (for example, `gpt-5.4`, `phi-4`) live directly on your project's own Foundry account and are reached through `client.get_openai_client()` — they are **not** represented as a separate `"AzureOpenAI"`-type connection here. That connection type only applies when attaching a *different, external* Azure OpenAI resource, which this workshop doesn't use. We validate that your deployed models are actually reachable with a live call in the next section, instead of searching for a connection that will never appear.

Listing the connections confirms that the components needed to build our AI applications are provisioned and reachable from your project.

In [ ]:
# List the properties of all connections in the project
connections = list(client.connections.list())
print(f"====> Listing of all connections (found {len(connections)}):")
for connection in connections:
    print(f"- {connection.name} | type={connection.type} | default={connection.is_default}")

## Validate Model and Search Connections
The cell below validates that we have properly provisioned and connected to:
1. **Azure AI Search**, via its project connection.
2. Your deployed **model** (`MODEL_DEPLOYMENT_NAME`), via a live call through the project's OpenAI-compatible client — since, as noted above, your own account's models aren't discoverable as a `connections.list()` entry.

Both of these are essential for the exercises ahead: the model provides the core language capabilities, while Azure AI Search enables efficient information retrieval and knowledge base functionality.


In [ ]:
# 1. Confirm the Azure AI Search connection exists
from azure.ai.projects.models import ConnectionType

search_conn_id = ""
for conn in client.connections.list():
    if conn.type == ConnectionType.AZURE_AI_SEARCH:
        search_conn_id = conn.id

print("====> Azure AI Search connection:")
if not search_conn_id:
    print("Not found - please create an Azure AI Search connection (see Exercise 1, Task 1).")
else:
    print(f"Found: {search_conn_id}")

# 2. Confirm the deployed model is actually reachable with a live call.
#    Own-account models don't show up in connections.list(), so this is the real signal
#    that MODEL_DEPLOYMENT_NAME and PROJECT_ENDPOINT are correctly configured.
#    Note: get_openai_client() needs OPENAI_API_VERSION set in your .env (see .env.example).
print("\n====> Azure OpenAI / model access (live check):")
try:
    openai_client = client.get_openai_client()
    model_name = os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-5.4")
    response = openai_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": "Say 'ready' and nothing else."}],
        max_completion_tokens=5,
    )
    print(f"Success - reached the '{model_name}' deployment: {response.choices[0].message.content!r}")
except Exception as e:
    print(f"Could not reach the model deployment: {e}")
    print("Check that MODEL_DEPLOYMENT_NAME matches a deployment shown as 'Succeeded' on the "
          "Build > Models page, that PROJECT_ENDPOINT is correct, and that OPENAI_API_VERSION "
          "is set in your .env.")